# Thresholds and Metrics

**Project question:** How can a probability model support an action decision without tuning on the test set?

By the end of this notebook, you should be able to:

- reserve validation data for threshold choice and test data for final evaluation
- compare precision, recall, specificity, and weighted decision cost
- interpret ROC-AUC separately from threshold and calibration quality

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay

In [ ]:
df = pd.read_csv(DATA / 'simulated_churn.csv')
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']
num = ['tenure_months', 'monthly_charge', 'support_tickets', 'usage_gb', 'satisfaction']
cat = ['contract', 'autopay']
X_development, X_test, y_development, y_test = train_test_split(
    X, y, stratify=y, random_state=4031, test_size=0.20
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_development, y_development, stratify=y_development, random_state=4031, test_size=0.25
)

def build_classifier():
    pre = ColumnTransformer([
        ('num', StandardScaler(), num),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat),
    ])
    return make_pipeline(pre, LogisticRegression(max_iter=1000, random_state=4031))

clf = build_classifier().fit(X_train, y_train)
valid_prob = clf.predict_proba(X_valid)[:, 1]
pd.Series({'overall_churn_rate': y.mean(), 'validation_churn_rate': y_valid.mean()})

Assume retention outreach is inexpensive and a missed churner is five times as costly as unnecessary outreach. We therefore choose the threshold on validation data by minimizing `FP + 5*FN`.

In [ ]:
def threshold_metrics(actual, probability, threshold):
    predicted = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    return {
        'threshold': threshold, 'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
        'precision': tp / max(tp + fp, 1),
        'recall': tp / max(tp + fn, 1),
        'specificity': tn / max(tn + fp, 1),
        'weighted_cost': fp + 5 * fn,
    }

validation_table = pd.DataFrame(
    threshold_metrics(y_valid, valid_prob, threshold)
    for threshold in np.arange(0.20, 0.81, 0.10)
)
validation_table

In [ ]:
chosen_threshold = float(
    validation_table.loc[validation_table['weighted_cost'].idxmin(), 'threshold']
)
final_clf = build_classifier().fit(X_development, y_development)
test_prob = final_clf.predict_proba(X_test)[:, 1]
final_metrics = pd.DataFrame([threshold_metrics(y_test, test_prob, chosen_threshold)])
final_metrics['roc_auc'] = roc_auc_score(y_test, test_prob)
final_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(test_prob[y_test.to_numpy() == 0], bins=10, alpha=0.65, label='observed 0')
axes[0].hist(test_prob[y_test.to_numpy() == 1], bins=10, alpha=0.65, label='observed 1')
axes[0].axvline(chosen_threshold, color='black', linestyle='--', label='chosen threshold')
axes[0].set_xlabel('predicted churn probability')
axes[0].legend()
RocCurveDisplay.from_predictions(y_test, test_prob, ax=axes[1])
axes[1].set_title('Final test ROC curve')
plt.tight_layout()

**Interpretation:** Threshold selection and final evaluation have separate data roles. ROC-AUC measures ranking, not the 5:1 cost and not calibration. In a real project, inspect calibration and monitor performance after deployment.

**Transfer exercise:** Specify one false-positive consequence and one false-negative consequence in your project. Propose a cost ratio or operational constraint, then state which data subset would be used to choose the threshold.